
**Dataset:** BTS On-Time Carrier Performance, 2025 (Jan–Mar)  
**Source:** https://www.transtats.bts.gov  

In [1]:
import os
import pandas as pd
import numpy as np

RAW_DIR = "."
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option("display.max_columns", None)

## Load Data

In [2]:
files = [
    "T_ONTIME_MARKETING2025_01.csv",
    "T_ONTIME_MARKETING2025_02.csv",
    "T_ONTIME_MARKETING2025_03.csv",
]

raw = pd.concat(
    [pd.read_csv(os.path.join(RAW_DIR, f)) for f in files],
    ignore_index=True
)
raw.head(3)

,YEAR,MONTH,DAY_OF_MONTH,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,1,1,9E,4780,10397,15370,820,-6.0,0.0,938,-7.0,0.0,0.0,0.0,137.0,116.0,674.0,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,9E,4780,15370,10397,1018,-1.0,0.0,1315,-23.0,0.0,0.0,0.0,95.0,81.0,674.0,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,9E,4781,10397,13230,1110,-4.0,0.0,1303,-10.0,0.0,0.0,0.0,107.0,83.0,620.0,NaN,NaN,NaN,NaN,NaN


In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1823522 entries, 0 to 1823521
Data columns (total 23 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY_OF_MONTH         int64  
 3   OP_UNIQUE_CARRIER    object 
 4   OP_CARRIER_FL_NUM    int64  
 5   ORIGIN_AIRPORT_ID    int64  
 6   DEST_AIRPORT_ID      int64  
 7   CRS_DEP_TIME         int64  
 8   DEP_DELAY            float64
 9   DEP_DEL15            float64
 10  CRS_ARR_TIME         int64  
 11  ARR_DELAY            float64
 12  ARR_DEL15            float64
 13  CANCELLED            float64
 14  DIVERTED             float64
 15  ACTUAL_ELAPSED_TIME  float64
 16  AIR_TIME             float64
 17  DISTANCE             float64
 18  CARRIER_DELAY        float64
 19  WEATHER_DELAY        float64
 20  NAS_DELAY            float64
 21  SECURITY_DELAY       float64
 22  LATE_AIRCRAFT_DELAY  float64
dtypes: float64(14), int64(8), object

In [4]:
# Missing value
missing = pd.DataFrame({
    "Missing"  : raw.isnull().sum(),
    "Missing %": (raw.isnull().mean() * 100).round(2)
})
missing[missing["Missing"] > 0]

,Missing,Missing %
DEP_DELAY,33744,1.85
DEP_DEL15,33744,1.85
ARR_DELAY,39125,2.15
ARR_DEL15,39125,2.15
ACTUAL_ELAPSED_TIME,39125,2.15
AIR_TIME,39125,2.15
CARRIER_DELAY,1471554,80.70
WEATHER_DELAY,1471554,80.70
NAS_DELAY,1471554,80.70
SECURITY_DELAY,1471554,80.70


In [5]:
# Duplicate rows
print("Duplicate rows:", raw.duplicated().sum())

# Cancelled / Diverted flights
print("CANCELLED:", raw["CANCELLED"].value_counts().to_dict())
print("DIVERTED :", raw["DIVERTED"].value_counts().to_dict())

Duplicate rows: 0
CANCELLED: {0.0: 1788660, 1.0: 34862}
DIVERTED : {0.0: 1819259, 1.0: 4263}


## Cleaning

In [6]:
df = raw.copy()

# Fix dtypes
df["CANCELLED"] = df["CANCELLED"].astype(int)
df["DIVERTED"]  = df["DIVERTED"].astype(int)

# Drop exact duplicates (none expected)
df.drop_duplicates(inplace=True)

# Replace NaNs with 0 for active on-time flights
delay_cause_cols = [
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"
]
active_ontime = (df["CANCELLED"] == 0) & (df["DIVERTED"] == 0) & (df["ARR_DEL15"] == 0)
df.loc[active_ontime, delay_cause_cols] = \
    df.loc[active_ontime, delay_cause_cols].fillna(0)

print(f"Rows after cleaning : {len(df):,}")

Rows after cleaning : 1,823,522


## new label

In [7]:
# Departure hour (0–23) from scheduled time
df["DEP_HOUR"] = df["CRS_DEP_TIME"] // 100

# Flight status label
def get_status(row):
    if row["CANCELLED"] == 1: return "Cancelled"
    if row["DIVERTED"]  == 1: return "Diverted"
    if row["ARR_DEL15"] == 1: return "Delayed"
    return "OnTime"

df["FLIGHT_STATUS"] = df.apply(get_status, axis=1).astype("category")

# Extreme delay flag: DEP_DELAY > 1000 min
active = (df["CANCELLED"] == 0) & (df["DIVERTED"] == 0)
df["IS_EXTREME_DELAY"] = ((active) & (df["DEP_DELAY"] > 1000)).astype(int)

# Total delay cause
df["TOTAL_DELAY_CAUSE"] = df[delay_cause_cols].sum(axis=1)
df.loc[(df["CANCELLED"] == 1) | (df["DIVERTED"] == 1), "TOTAL_DELAY_CAUSE"] = np.nan

## Summary

In [8]:
# Flight status distribution
status = df["FLIGHT_STATUS"].value_counts().reset_index()
status.columns = ["Status", "Count"]
status["Pct %"] = (status["Count"] / len(df) * 100).round(2)
status

,Status,Count,Pct %
0,OnTime,1432429,78.55
1,Delayed,351968,19.30
2,Cancelled,34862,1.91
3,Diverted,4263,0.23


In [9]:
df.head(3)

,YEAR,MONTH,DAY_OF_MONTH,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DEP_HOUR,FLIGHT_STATUS,IS_EXTREME_DELAY,TOTAL_DELAY_CAUSE
0,2025,1,1,9E,4780,10397,15370,820,-6.0,0.0,938,-7.0,0.0,0,0,137.0,116.0,674.0,0.0,0.0,0.0,0.0,0.0,8,OnTime,0,0.0
1,2025,1,1,9E,4780,15370,10397,1018,-1.0,0.0,1315,-23.0,0.0,0,0,95.0,81.0,674.0,0.0,0.0,0.0,0.0,0.0,10,OnTime,0,0.0
2,2025,1,1,9E,4781,10397,13230,1110,-4.0,0.0,1303,-10.0,0.0,0,0,107.0,83.0,620.0,0.0,0.0,0.0,0.0,0.0,11,OnTime,0,0.0


## Drop Columns Not Needed for predict

In [ ]:
# YEAR                 constant (all 2025), zero variance
# OP_CARRIER_FL_NUM    flight number just an ID, not a feature
# DEP_DEL15            redundant: just DEP_DELAY >= 15
# TOTAL_DELAY_CAUSE    redundant: sum of the five cause columns
# ACTUAL_ELAPSED_TIME  only known after landing
# AIR_TIME             only known after landing
# CARRIER_DELAY, WEATHER_DELAY, NAS_DELAY,SECURITY_DELAY, LATE_AIRCRAFT_DELAY    only recorded after a delay occurs

drop_cols = [
    "YEAR",
    "OP_CARRIER_FL_NUM",
    "DEP_DEL15",
    "TOTAL_DELAY_CAUSE",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

df.drop(columns=drop_cols, inplace=True)
df.head(3)

,MONTH,DAY_OF_MONTH,OP_UNIQUE_CARRIER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,CRS_DEP_TIME,DEP_DELAY,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,DISTANCE,DEP_HOUR,FLIGHT_STATUS,IS_EXTREME_DELAY
0,1,1,9E,10397,15370,820,-6.0,938,-7.0,0.0,0,0,674.0,8,OnTime,0
1,1,1,9E,15370,10397,1018,-1.0,1315,-23.0,0.0,0,0,674.0,10,OnTime,0
2,1,1,9E,10397,13230,1110,-4.0,1303,-10.0,0.0,0,0,620.0,11,OnTime,0


## Export cleaned data

In [11]:
# Raw combined
raw.to_csv(os.path.join(OUT_DIR, "flight_delay_raw_2025_Q1.csv"), index=False)

# Cleaned dataset
df.to_csv(os.path.join(OUT_DIR, "flight_delay_clean_2025_Q1.csv"), index=False)